# Foundation-model YOLOv8 — text-line detection (CB55)

Same task pattern as the UNet notebook, but using YOLOv8 bounding boxes for detection.
The encoder is a DINOv2/DINOv3/RADIO hierarchical encoder via `hier_encoder`.

Because Ultralytics YOLOv8 defines its models programmatically and requires specific feature map 
shapes, we use `hier_encoder.adapters.build_trainable_yolo` to dynamically assemble a YOLO 
PANet neck and Detect head around our vision foundation models. This outputs a standard 
`ultralytics.YOLO` object that can seamlessly hook into Ultralytics' training and validation engines.

**Sections**
1. Knobs & YOLOv8 build
2. Train + Validate (using YOLOv8 trainer)
3. Inference examples

## 1. Knobs & YOLOv8 build

In [1]:
import os
import torch
from hier_encoder import EncoderConfig
from hier_encoder.adapters import build_trainable_yolo

# ---- CHOOSE ENCODER FAMILY ----------------------------------------------------
ENCODER    = 'dinov3'    # 'dinov2' | 'dinov2_reg' | 'dinov3' | 'am-radio'
FREEZE     = True        # frozen ViT few-shot regime; False -> full fine-tune
LORA_RANK  = 0           # 0 off; 4/8/16 -> LoRA on q/v
STRATEGY   = 'sfp'       # 'sfp' | 'multiblock'
# --------------------------------------------------------------------------------
_ENC_BACKBONE = {
    'dinov2':     'dinov2_vitb14',
    'dinov2_reg': 'dinov2_vitb14_reg',
    'dinov3':     'dinov3_vitb16',
    'am-radio':   'radio_v2.5-b',
}

YOLO_SCALE  = 's'
NUM_CLASSES = 1
P2_LEVEL    = True  # P2 level (stride 4) is helpful for thin objects like manuscript text lines

CROP      = 640
BATCH     = 4
EPOCHS    = 30

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = EncoderConfig(
    backbone=_ENC_BACKBONE[ENCODER],
    pretrained=True,
    freeze_backbone=FREEZE,
    feature_strategy=STRATEGY,
    lora_rank=LORA_RANK,
)

print(f"Building YOLOv8-{YOLO_SCALE} with {ENCODER} backbone...")
# build_trainable_yolo creates a real `ultralytics.YOLO` whose backbone is the hier encoder.
model = build_trainable_yolo(cfg, nc=NUM_CLASSES, scale=YOLO_SCALE, p2=P2_LEVEL)

tr = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.model.parameters())
print(f"params: total={tot/1e6:.1f}M trainable={tr/1e6:.1f}M")

Building YOLOv8-s with dinov3 backbone...


Using cache found in /home/artur/.cache/torch/hub/facebookresearch_dinov3_main


Downloading: "https://dl.fbaipublicfiles.com/dinov3/dinov3_vitb16/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth" to /home/artur/.cache/torch/hub/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth


/home/artur/Thesis/fsl-tl-manuscripts/hier_encoder/build.py:47: UserWarning: could not load pretrained 'dinov3_vitb16' (HTTPError: HTTP Error 403: Forbidden); using a random-init fallback ViT with matching dims. Set DINOV3_WEIGHTS_DIR or check the network for real weights.
  self.backbone: BackboneWrapper = load_backbone(cfg)


params: total=125.6M trainable=36.8M


## 2. Train + Validate

We use Ultralytics' built-in `.train()` directly. It takes our dataset YAML and handles the train/val loop, augmentations, EMA, and metrics calculation implicitly.

In [2]:
DATA_YAML = "00_data/DIVA-HisDB/yolo_dataset_CB55/diva_cb55_det.yaml"

if not os.path.exists(DATA_YAML):
    print(f"Warning: {DATA_YAML} not found. Ensure dataset is prepared.")
else:
    os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
    # Start training! Check `runs/detect/` for tensorboard/results.
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=CROP,
        batch=BATCH,
        device="0" if DEVICE == 'cuda' else "cpu",
        project=f"runs/hier_yolov8_{ENCODER}_{STRATEGY}",
        name="train_cb55"
    )
    # Note: validation is run automatically at the end of each epoch.

New https://pypi.org/project/ultralytics/8.4.102 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7785MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=00_data/DIVA-HisDB/yolo_dataset_CB55/diva_cb55_det.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/tmp/tmpbux57gl0.yaml, moment

Using cache found in /home/artur/.cache/torch/hub/facebookresearch_dinov3_main


Downloading: "https://dl.fbaipublicfiles.com/dinov3/dinov3_vitb16/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth" to /home/artur/.cache/torch/hub/checkpoints/dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth


/home/artur/Thesis/fsl-tl-manuscripts/hier_encoder/build.py:47: UserWarning: could not load pretrained 'dinov3_vitb16' (HTTPError: HTTP Error 403: Forbidden); using a random-init fallback ViT with matching dims. Set DINOV3_WEIGHTS_DIR or check the network for real weights.
  self.backbone: BackboneWrapper = load_backbone(cfg)


  0                  -1  1 119336128  hier_encoder.adapters.yolo_adapter.HierEncoderYOLOParse['sc_s', True]                
  1                   0  1         0  ultralytics.nn.modules.conv.Index            [0]                           
  2                   0  1         0  ultralytics.nn.modules.conv.Index            [1]                           
  3                   0  1         0  ultralytics.nn.modules.conv.Index            [2]                           
  4                   0  1         0  ultralytics.nn.modules.conv.Index            [3]                           
  5                   4  1    656896  ultralytics.nn.modules.block.SPPF            [512, 512, 5]                 
  6                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None, 2, 'nearest']          
  7             [-1, 3]  1         0  ultralytics.nn.modules.conv.Concat           [1]                           
  8                  -1  1    591360  ultralytics.nn.modules.block.C2f        

...

Exception in thread Thread-8 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/artur/Thesis/fsl-tl-manuscripts/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/artur/Thesis/fsl-tl-manuscripts/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/artur/Thesis/fsl-tl-manuscripts/.venv/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_storage_

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 71 weight(decay=0.0), 127 weight(decay=0.0005), 149 bias(decay=0.0)
: 0% ──────────── 0/10  5.0s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
WARNING ⚠️ CUDA out of memory with batch=1. Reducing to batch=1 and retrying (3/3).
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3628.8±4192.8 MB/s, size: 5194.1 KB)
train: Scanning /home/artur/Thesis/fsl-tl-manuscripts/00_data/DIVA-HisDB/yolo_dataset_CB55/labels/train.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 2.5Mit/s 0.0s
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0

Exception in thread Thread-10 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/home/artur/Thesis/fsl-tl-manuscripts/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/home/artur/Thesis/fsl-tl-manuscripts/.venv/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/artur/Thesis/fsl-tl-manuscripts/.venv/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 541, in rebuild_storage

optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 71 weight(decay=0.0), 127 weight(decay=0.0005), 149 bias(decay=0.0)
: 0% ──────────── 0/20  3.2s

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
: 0% ──────────── 0/20  0.1s


OutOfMemoryError: CUDA out of memory. Tried to allocate 12.00 MiB. GPU 0 has a total capacity of 7.60 GiB of which 52.12 MiB is free. Process 36305 has 4.17 GiB memory in use. Including non-PyTorch memory, this process has 1.65 GiB memory in use. Of the allocated memory 1.41 GiB is allocated by PyTorch, and 99.19 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 3. Inference

Run batched full-resolution testing over validation/test splits.

In [ ]:
import glob
val_images = sorted(glob.glob("00_data/DIVA-HisDB/yolo_dataset_CB55/images/val/*.jpg"))[:4]

if val_images:
    print(f"Running inference on {len(val_images)} images...")
    preds = model.predict(val_images, imgsz=CROP, save=True, conf=0.25)
    print(f"\nPredictions saved to: {preds[0].save_dir}")
